# 140 · Audio Agent — Transcribe, Classify, Route

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/140-audio-agent/audio_agent_workbook.ipynb)

**Source:** `examples/140-audio-agent/`

## What you'll learn

- How OpenAI Whisper transcribes audio files to plain text in a single API call
- How to use a JSON-schema-in-prompt to force a structured, enumerable output from GPT-5.4 nano
- How a dictionary-based routing layer eliminates LLM calls for deterministic dispatch
- How LangGraph `StateGraph` threads immutable state through a three-node audio pipeline
- How to handle edge cases: empty transcripts, ambiguous intents, and non-English audio

## Requirements

- **Part A** (concept demonstrations): no API key needed — runs entirely on CPU
- **Part B** (live pipeline): requires `OPENAI_API_KEY` in a `.env` file and a `.wav` audio file

In [ ]:
%pip install -q openai python-dotenv langgraph

---
## Part A — CPU-Safe Concept Demonstrations

No API key needed. All cells in Part A run offline using mock data and pure Python logic.

### Whisper Model Comparison

| Model | Parameters | WER (English) | Speed (relative) | VRAM | Best for |
|-------|-----------|---------------|-----------------|------|----------|
| tiny | 39M | ~5.7% | 32x | ~1 GB | Edge, real-time, mobile |
| base | 74M | ~4.2% | 16x | ~1 GB | Batch on CPU |
| small | 244M | ~3.4% | 6x | ~2 GB | Balanced |
| medium | 769M | ~3.1% | 2x | ~5 GB | High accuracy |
| large-v3 | 1550M | ~2.7% | 1x | ~10 GB | Best quality, GPU only |

**Note:** `whisper-1` via the OpenAI API is approximately large-v2 quality. WER = Word Error Rate (lower is better).

### Audio Preprocessing Requirements

- Sample rate: 16 kHz (Whisper's native rate; other rates are resampled internally)
- Channels: mono (stereo is averaged to mono)
- Formats: WAV, MP3, MP4, M4A, FLAC, OGG, WebM all accepted by OpenAI API
- Duration: max 25 MB file size via API; for longer audio, chunk at silence gaps

| Issue | Effect on ASR | Fix |
|-------|--------------|-----|
| 44.1kHz stereo | Extra decode step | Convert: `ffmpeg -ar 16000 -ac 1` |
| Low volume | Missed words | Normalize: `ffmpeg -af loudnorm` |
| Background noise | Increased WER | Denoise: `ffmpeg -af anlmdn` |
| MP3 artifacts | Codec artifacts | Re-encode or use WAV |

### Intent Classification Approaches

| Approach | How it works | Pros | Cons |
|----------|-------------|------|------|
| Regex rules | Pattern match keywords | Instant, no API cost, deterministic | Brittle, misses paraphrases |
| Embedding similarity | Embed transcript, cosine sim to intent vectors | Handles synonyms, fast inference | Needs embedding API or local model |
| LLM prompt | Send transcript to gpt-5.4-nano with JSON schema | Flexible, handles edge cases, extracts metadata | API cost, latency ~500ms |

**This agent uses approach 3 (LLM prompt)** for maximum flexibility.

### Pipeline Architecture

```
Audio file (.wav / .mp3)
      │
      ▼
┌──────────────────┐
│  transcribe_node │  Whisper-1 → plain text transcript
└─────────┬────────┘
          │  transcript: str
          ▼
┌──────────────────┐
│   analyze_node   │  gpt-5.4-nano → JSON {intent, urgency, sentiment, ...}
└─────────┬────────┘
          │  analysis: dict
          ▼
┌──────────────────┐
│    route_node    │  ROUTING_MAP lookup → queue name
└─────────┬────────┘
          │  queue: str
          ▼
      [END]

ROUTING_MAP:
  billing      → billing-queue
  technical    → tier1-support
  complaint    → customer-success
  cancellation → retention-team
  general      → general-support
```

### LangGraph `AudioState` — Immutable State Flow

LangGraph passes a **TypedDict** through each node. Every node receives the full state
and returns a new copy (via `{**state, key: new_value}`). The original is never mutated.

```python
class AudioState(TypedDict):
    audio_path: str    # set once at invocation, never changed
    transcript: str    # populated by transcribe_node
    analysis: dict     # populated by analyze_node
    queue: str         # populated by route_node
```

**State evolution through the graph:**

| After node | `audio_path` | `transcript` | `analysis` | `queue` |
|---|---|---|---|---|
| Initial input | `/tmp/sample.wav` | `""` | `{}` | `""` |
| `transcribe_node` | unchanged | `"Hi, I was charged..."` | `{}` | `""` |
| `analyze_node` | unchanged | unchanged | `{"intent": "billing", ...}` | `""` |
| `route_node` | unchanged | unchanged | unchanged | `"billing-queue"` |

Each node only adds to the state — it never overwrites prior fields unless explicitly returning them.

In [ ]:
# Demonstrate how AudioState evolves without mutation
# Pure Python — no LangGraph import needed to understand the pattern

from typing import TypedDict

class AudioState(TypedDict):
    audio_path: str
    transcript: str
    analysis: dict
    queue: str

# Initial state (as passed to workflow.invoke)
initial: AudioState = {"audio_path": "/tmp/sample.wav", "transcript": "", "analysis": {}, "queue": ""}

# Simulate transcribe_node
after_transcribe: AudioState = {**initial, "transcript": "Hi, I was charged twice this month and need a refund."}

# Simulate analyze_node
after_analyze: AudioState = {**after_transcribe, "analysis": {"intent": "billing", "urgency": "high", "sentiment": "negative", "action_required": "Issue refund"}}

# Simulate route_node
after_route: AudioState = {**after_analyze, "queue": "billing-queue"}

# Show the evolution
for label, state in [("initial", initial), ("after_transcribe", after_transcribe), ("after_analyze", after_analyze), ("after_route", after_route)]:
    print(f"{label}:")
    print(f"  transcript : {repr(state['transcript'][:40])}")
    print(f"  analysis   : {state['analysis']}")
    print(f"  queue      : {repr(state['queue'])}")
    print()

# Key insight: initial is never modified — each assignment creates a new dict
print(f"initial is unchanged: transcript={repr(initial['transcript'])}, queue={repr(initial['queue'])}")

In [ ]:
# Intent examples by category — all 5 intent types with sample transcripts

INTENT_EXAMPLES = {
    "billing": {
        "transcript": "Hi, I was charged twice this month and I need a refund for the duplicate charge.",
        "expected_route": "billing-queue",
        "urgency": "high",
    },
    "technical": {
        "transcript": "My dashboard keeps crashing whenever I try to export a report as PDF.",
        "expected_route": "tier1-support",
        "urgency": "medium",
    },
    "general": {
        "transcript": "I just wanted to know what your office hours are and whether you offer phone support.",
        "expected_route": "general-support",
        "urgency": "low",
    },
    "complaint": {
        "transcript": "This is unacceptable. I've been waiting three weeks for someone to fix my account and nothing has happened.",
        "expected_route": "customer-success",
        "urgency": "high",
    },
    "cancellation": {
        "transcript": "I'd like to cancel my subscription. The product doesn't meet my team's needs anymore.",
        "expected_route": "retention-team",
        "urgency": "medium",
    },
}

print(f"{'Intent':<14} {'Route':<22} {'Urgency':<8} {'Transcript preview'}")
print("-" * 80)
for intent, data in INTENT_EXAMPLES.items():
    preview = data["transcript"][:45] + "..."
    print(f"{intent:<14} {data['expected_route']:<22} {data['urgency']:<8} {preview}")

In [ ]:
# Classification prompt construction — builds and displays the full prompt
# No API call — just shows what gets sent to gpt-5.4-nano

def build_classification_prompt(transcript: str) -> str:
    return f"""Analyze this support call transcript and respond with JSON only:
{{
  "intent": "billing|technical|general|complaint|cancellation",
  "urgency": "low|medium|high",
  "product_mentioned": "string or null",
  "action_required": "string describing next step",
  "sentiment": "positive|neutral|negative"
}}

Transcript: {transcript}"""


# Demo with the billing sample transcript
sample = INTENT_EXAMPLES["billing"]["transcript"]
prompt = build_classification_prompt(sample)
print("=== Classification Prompt ===")
print(prompt)
print(f"\nPrompt length: {len(prompt)} chars, ~{len(prompt)//4} tokens")

In [ ]:
# Routing logic demo — given intent JSON, show which queue fires
import json

ROUTING_MAP = {
    "billing": "billing-queue",
    "technical": "tier1-support",
    "complaint": "customer-success",
    "cancellation": "retention-team",
    "general": "general-support",
}


def route(intent: str) -> str:
    return ROUTING_MAP.get(intent, "general-support")


# Simulate analysis results for each intent
mock_analyses = [
    {"intent": "billing", "urgency": "high", "sentiment": "negative", "action_required": "Issue refund"},
    {"intent": "technical", "urgency": "medium", "sentiment": "neutral", "action_required": "Escalate to Tier 1"},
    {"intent": "cancellation", "urgency": "medium", "sentiment": "negative", "action_required": "Offer retention deal"},
    {"intent": "complaint", "urgency": "high", "sentiment": "negative", "action_required": "Immediate callback"},
    {"intent": "general", "urgency": "low", "sentiment": "positive", "action_required": "Send FAQ link"},
]

print(f"{'Intent':<14} {'Queue':<22} {'Urgency':<10} {'Action'}")
print("-" * 75)
for a in mock_analyses:
    q = route(a["intent"])
    print(f"{a['intent']:<14} {q:<22} {a['urgency']:<10} {a['action_required']}")

print(f"\nFallback: route('unknown') → '{route('unknown')}'")

In [ ]:
# Edge case simulation — shows how the system handles problematic inputs

EDGE_CASES = [
    {
        "name": "Empty transcript",
        "transcript": "",
        "problem": "Whisper returns empty string (silence or non-speech audio)",
        "mitigation": "Check len(transcript.strip()) == 0 before calling classify",
    },
    {
        "name": "Ambiguous intent",
        "transcript": "I need help",
        "problem": "Too vague for reliable classification",
        "mitigation": "Default to 'general' intent with low confidence flag",
    },
    {
        "name": "Multi-intent",
        "transcript": "I want to cancel but also get a refund for last month first",
        "problem": "Two intents: cancellation + billing",
        "mitigation": "Route to highest-priority: cancellation (retention team handles refund too)",
    },
    {
        "name": "Non-English",
        "transcript": "Necesito ayuda con mi factura",
        "problem": "Whisper transcribes correctly but routing was trained on English intents",
        "mitigation": "Add language detection; route to multilingual-support queue",
    },
]

print("=== Edge Case Handling ===\n")
for ec in EDGE_CASES:
    print(f"Case: {ec['name']}")
    print(f"  Input:      '{ec['transcript'][:60]}'")
    print(f"  Problem:    {ec['problem']}")
    print(f"  Mitigation: {ec['mitigation']}")
    print()

### Robust JSON Parsing from LLM Output

LLMs sometimes wrap JSON in markdown fences or add trailing prose, even when instructed not to.
A production-grade parser handles these gracefully rather than crashing:

| LLM output variant | Naive `json.loads()` | Robust parser |
|---|---|---|
| `{"intent": "billing"}` | OK | OK |
| `` ```json\n{...}\n``` `` | FAIL | OK (strip fences) |
| `{"intent": "billing"}\n\nNote: ...` | FAIL | OK (slice to last `}`) |
| `Here's the JSON: {"intent": ...}` | FAIL | OK (find first `{`) |
| `Sorry, I can't classify this` | FAIL | Returns fallback dict |

The source code uses `.strip().strip("\`\`\`json").strip("\`\`\`")` as a minimal guard.
For production, a more thorough extractor is worth adding.

In [ ]:
# Robust JSON extraction from LLM responses
import json, re

def safe_parse_llm_json(text: str, fallback: dict = None) -> dict:
    """Extract JSON from an LLM response string, handling common formatting quirks."""
    if fallback is None:
        fallback = {"intent": "general", "urgency": "low", "sentiment": "neutral", "action_required": "Parsing failed"}
    # Strip markdown code fences
    text = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.MULTILINE)
    text = re.sub(r'```\s*$', '', text.strip(), flags=re.MULTILINE)
    text = text.strip()
    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Try extracting the first JSON object
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return fallback

# Test all 5 variants
test_cases = [
    ('{"intent": "billing", "urgency": "high"}',           "clean JSON"),
    ('```json\n{"intent": "technical"}\n```',               "fenced JSON"),
    ('{"intent": "general"}\n\nNote: very vague transcript', "JSON + trailing prose"),
    ('Here is the classification: {"intent": "complaint"}',  "prose + JSON"),
    ('I cannot classify this transcript.',                    "no JSON at all"),
]

for raw, label in test_cases:
    result = safe_parse_llm_json(raw)
    print(f"[{label}]")
    print(f"  Input  : {repr(raw[:50])}")
    print(f"  Output : {result}")
    print()

---
### Exercise 1: Add a Human Escalation Intent

The `ROUTING_MAP` currently handles five intents. Add a sixth intent `"escalation"` that routes to `"human-agent-queue"`.

Example transcripts that should trigger `"escalation"`:
- *"Let me speak to a human right now."*
- *"I want to talk to a manager."*
- *"Stop giving me automated responses and connect me to a real person."*

You need to update **two things**:
1. `ROUTING_MAP` — add the new intent → queue mapping
2. The classification prompt — add `"escalation"` to the `intent` enum

Try it in the TODO cell before looking at the answer key.

In [ ]:
# Exercise 1: Add a "human escalation" intent
# Transcripts like "Let me speak to a human" or "I want to talk to a manager"
# should route to "human-agent-queue"

# TODO: Add "escalation" to ROUTING_MAP
ROUTING_MAP_V2 = {
    "billing": "billing-queue",
    "technical": "tier1-support",
    "complaint": "customer-success",
    "cancellation": "retention-team",
    "general": "general-support",
    # TODO: add escalation intent here
}

# TODO: Update the intent list in the classification prompt
# The JSON schema line currently reads:
# "intent": "billing|technical|general|complaint|cancellation"
# Update it to include "escalation"

def build_classification_prompt_v2(transcript: str) -> str:
    # TODO: update the intent options in the prompt
    pass

# Test
test_transcript = "I've tried three times to fix this and nothing works. I want to speak to a manager now."
# expected: intent = "escalation", route = "human-agent-queue"

In [ ]:
# Answer Key — Exercise 1

ROUTING_MAP_V2 = {
    "billing": "billing-queue",
    "technical": "tier1-support",
    "complaint": "customer-success",
    "cancellation": "retention-team",
    "general": "general-support",
    "escalation": "human-agent-queue",  # NEW
}


def build_classification_prompt_v2(transcript: str) -> str:
    return f"""Analyze this support call transcript and respond with JSON only:
{{
  "intent": "billing|technical|general|complaint|cancellation|escalation",
  "urgency": "low|medium|high",
  "product_mentioned": "string or null",
  "action_required": "string describing next step",
  "sentiment": "positive|neutral|negative"
}}

Classification guide for the new intent:
- escalation: customer explicitly requests a human agent or manager

Transcript: {transcript}"""


def route_v2(intent: str) -> str:
    return ROUTING_MAP_V2.get(intent, "general-support")


# Verify the new routing
test_cases = [
    ("escalation", "human-agent-queue"),
    ("billing", "billing-queue"),
    ("unknown_intent", "general-support"),
]
print("Routing verification:")
for intent, expected in test_cases:
    actual = route_v2(intent)
    status = "PASS" if actual == expected else "FAIL"
    print(f"  [{status}] '{intent}' -> '{actual}' (expected '{expected}')")
print()
print("Key insight: update both the prompt AND the routing map to keep them in sync.")
print("A schema validation step (e.g. pydantic) would catch drift between the two.")

---
### Exercise 2: Add Confidence Scoring

The current `analysis` dict has no measure of how certain the model is about its classification.

**Your task:** Add a `"confidence"` field (float, 0.0–1.0) to the JSON schema in the prompt. Then use the confidence value to decide whether to route automatically or flag for human review.

Routing logic:
- `confidence >= 0.7` → route normally using `ROUTING_MAP`
- `confidence < 0.7` → route to `"review-queue"` regardless of intent

Hint: update the prompt and add a threshold check after parsing the JSON. No graph changes needed.

In [ ]:
# Exercise 2: Add confidence scoring with threshold-based routing

CONFIDENCE_THRESHOLD = 0.7


def build_classification_prompt_v3(transcript: str) -> str:
    # TODO: Add a "confidence" field (0.0-1.0) to the JSON schema
    return f"""Analyze this support call transcript and respond with JSON only:
{{
  "intent": "billing|technical|general|complaint|cancellation",
  "urgency": "low|medium|high",
  "product_mentioned": "string or null",
  "action_required": "string describing next step",
  "sentiment": "positive|neutral|negative"
}}

Transcript: {transcript}"""


def route_with_confidence(analysis: dict) -> str:
    # TODO: If confidence < CONFIDENCE_THRESHOLD, return "review-queue"
    # Otherwise, use the normal ROUTING_MAP lookup
    pass


# Test cases
test_analyses = [
    {"intent": "billing", "confidence": 0.95},   # -> billing-queue
    {"intent": "technical", "confidence": 0.50},  # -> review-queue
    {"intent": "general", "confidence": 0.71},    # -> general-support
]

In [ ]:
# Answer Key — Exercise 2

CONFIDENCE_THRESHOLD = 0.7


def build_classification_prompt_v3(transcript: str) -> str:
    return f"""Analyze this support call transcript and respond with JSON only:
{{
  "intent": "billing|technical|general|complaint|cancellation",
  "urgency": "low|medium|high",
  "product_mentioned": "string or null",
  "action_required": "string describing next step",
  "sentiment": "positive|neutral|negative",
  "confidence": 0.0
}}

Set confidence to a float between 0.0 and 1.0 based on how certain you are
about the intent classification.

Transcript: {transcript}"""


def route_with_confidence(analysis: dict) -> str:
    confidence = analysis.get("confidence", 0.0)
    if confidence < CONFIDENCE_THRESHOLD:
        return "review-queue"
    intent = analysis.get("intent", "general")
    return ROUTING_MAP.get(intent, "general-support")


# Verify confidence-gated routing
test_analyses = [
    {"intent": "billing", "confidence": 0.95},
    {"intent": "technical", "confidence": 0.50},
    {"intent": "general", "confidence": 0.71},
    {"intent": "cancellation", "confidence": 0.69},
]
expected_routes = ["billing-queue", "review-queue", "general-support", "review-queue"]

print("Confidence-gated routing verification:")
print(f"{'Intent':<14} {'Confidence':<12} {'Route':<22} {'Status'}")
print("-" * 65)
for analysis, expected in zip(test_analyses, expected_routes):
    actual = route_with_confidence(analysis)
    status = "PASS" if actual == expected else "FAIL"
    print(f"{analysis['intent']:<14} {analysis['confidence']:<12} {actual:<22} [{status}]")

print("\nKey insight: threshold filtering is separate from routing.")
print("Tune the threshold without touching ROUTING_MAP.")

### API Cost Estimation for the Audio Pipeline

Each call through the full pipeline uses two OpenAI endpoints:

| Step | Model | Pricing (as of 2024) |
|---|---|---|
| Transcription | `whisper-1` | $0.006 / minute of audio |
| Classification | `gpt-5.4-nano` input | Check the current provider pricing |
| Classification | `gpt-5.4-nano` output | Check the current provider pricing |

A typical 2-minute support call with a ~200-token transcript and ~150-token prompt costs:
- Whisper: 2 min × $0.006 = **$0.012**
- GPT-4o-mini input (~350 tokens): $0.000053
- GPT-4o-mini output (~100 tokens): $0.000060
- **Total: ~$0.012 per call** — roughly $12 per 1,000 calls

For high-volume use cases, consider: (1) batching Whisper calls, (2) caching identical transcripts, (3) using regex pre-filtering to skip LLM classification for obvious intents.

In [ ]:
# Cost estimation for the audio pipeline at various call volumes

WHISPER_PER_MINUTE = 0.006        # USD per minute of audio
GPT4O_MINI_INPUT_PER_1M = 0.15   # USD per 1M input tokens
GPT4O_MINI_OUTPUT_PER_1M = 0.60  # USD per 1M output tokens

def estimate_pipeline_cost(
    num_calls: int,
    avg_duration_minutes: float = 2.0,
    avg_transcript_tokens: int = 200,
    prompt_overhead_tokens: int = 150,
    avg_output_tokens: int = 100,
) -> dict:
    whisper_cost = num_calls * avg_duration_minutes * WHISPER_PER_MINUTE
    total_input_tokens = num_calls * (avg_transcript_tokens + prompt_overhead_tokens)
    total_output_tokens = num_calls * avg_output_tokens
    gpt_input_cost = (total_input_tokens / 1_000_000) * GPT4O_MINI_INPUT_PER_1M
    gpt_output_cost = (total_output_tokens / 1_000_000) * GPT4O_MINI_OUTPUT_PER_1M
    total = whisper_cost + gpt_input_cost + gpt_output_cost
    return {"calls": num_calls, "whisper_usd": round(whisper_cost, 4), "gpt_usd": round(gpt_input_cost + gpt_output_cost, 4), "total_usd": round(total, 4)}

print(f"{'Calls':>10} | {'Whisper':>10} | {'GPT-4o-mini':>12} | {'Total':>10}")
print("-" * 52)
for n in [1, 100, 1_000, 10_000, 100_000]:
    est = estimate_pipeline_cost(n)
    print(f"{est['calls']:>10,} | ${est['whisper_usd']:>9.4f} | ${est['gpt_usd']:>11.4f} | ${est['total_usd']:>9.4f}")

print()
print("Optimization tip: regex pre-filter obvious intents to skip the GPT-4o-mini call.")
print("If 40% of calls are 'general' (easily matched by keywords), you save ~40% on LLM cost.")

---
## Part B — Live Pipeline (requires OPENAI_API_KEY + a .wav file)

**Requirements:**
- `OPENAI_API_KEY` set in a `.env` file
- A speech `.wav` audio file selected with `AUDIO_AGENT_AUDIO_PATH`

This section deliberately requires speech rather than generating a tone: an empty transcription would not verify intent classification or queue routing.

**To create a real speech WAV on macOS:**
```bash
say -o /tmp/sample.aiff "Hi, I was charged twice this month and need a refund."
afconvert -f WAVE -d LEI16@16000 /tmp/sample.aiff /tmp/sample.wav
export AUDIO_AGENT_AUDIO_PATH=/tmp/sample.wav
```

In [ ]:
# Part B setup — fail fast on the key and a real speech WAV

import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Add it to your .env file or set it in the environment.\n"
        "Part B requires a valid OpenAI API key."
    )

WAV_PATH = os.environ.get("AUDIO_AGENT_AUDIO_PATH")
if not WAV_PATH or not os.path.isfile(WAV_PATH):
    raise FileNotFoundError(
        "Set AUDIO_AGENT_AUDIO_PATH to a readable speech WAV before running Part B."
    )
size = os.path.getsize(WAV_PATH)
print(f"Using speech WAV {WAV_PATH}: {size:,} bytes")

print("\nSetup complete — ready for Part B")

In [ ]:
# Part B — execute the production graph from src/, not a notebook copy

import json
from openai import OpenAI
from src.workflow import create_workflow

client = OpenAI(api_key=api_key)
workflow = create_workflow()
cfg = {"configurable": {"client": client, "model": "gpt-5.4-nano"}}
result = workflow.invoke(
    {"audio_path": WAV_PATH, "transcript": "", "analysis": {}, "queue": ""},
    config=cfg,
)

print(f"\n=== Final Result ===")
print(f"Audio file : {result['audio_path']}")
transcript_preview = result['transcript'][:80] + ("..." if len(result['transcript']) > 80 else "")
print(f"Transcript : {transcript_preview or '(empty)'}")
print(f"Intent     : {result['analysis'].get('intent', 'N/A')}")
print(f"Urgency    : {result['analysis'].get('urgency', 'N/A')}")
print(f"Queue      : {result['queue']}")

# This fixture is deliberately a spoken billing request; verify every live stage.
assert result["transcript"].strip(), "Whisper returned no speech"
assert result["analysis"]["intent"] == "billing", result["analysis"]
assert result["queue"] == "billing-queue", result["queue"]
print("LIVE ROUTING CONTRACT: PASS")

### Extending to Multi-Language Support

Whisper automatically detects the spoken language and sets `result.language` on the
transcription response. You can use this to route non-English calls before classification:

```python
result = client.audio.transcriptions.create(
    model="whisper-1",
    file=f,
    response_format="verbose_json",  # includes language field
)
language = result.language  # e.g. "spanish", "french", "mandarin"
```

**Language-aware routing:**

| Detected language | Routing strategy |
|---|---|
| English | Normal classify → ROUTING_MAP |
| Spanish | Translate transcript, then classify; or route directly to ES-support queue |
| French | Same as Spanish |
| Other | Route to multilingual-support for human agent |

Using `verbose_json` response format also returns `segments` (with timestamps) and `words`,
which is useful for call quality metrics like speaking pace and silence ratio.

In [ ]:
# Demonstrate the different Whisper response formats (no API call — structure only)
# This shows what fields are available when using verbose_json vs plain text

# Plain text response (default, response_format='text'):
plain_text_response = "Hi, I was charged twice this month and need a refund."

# Verbose JSON response structure (response_format='verbose_json'):
verbose_json_mock = {
    "task": "transcribe",
    "language": "english",
    "duration": 4.2,
    "text": "Hi, I was charged twice this month and need a refund.",
    "segments": [
        {"id": 0, "start": 0.0, "end": 2.1, "text": " Hi, I was charged twice this month", "no_speech_prob": 0.01},
        {"id": 1, "start": 2.1, "end": 4.2, "text": " and need a refund.", "no_speech_prob": 0.02},
    ],
}

print("=== Plain text response ===")
print(repr(plain_text_response))
print()

print("=== Verbose JSON response fields ===")
for key, val in verbose_json_mock.items():
    if key != "segments":
        print(f"  {key}: {repr(val)}")
print(f"  segments: {len(verbose_json_mock['segments'])} segment(s)")
print()

# Derive quality metrics from verbose response
avg_no_speech = sum(s['no_speech_prob'] for s in verbose_json_mock['segments']) / len(verbose_json_mock['segments'])
words_per_minute = len(verbose_json_mock['text'].split()) / (verbose_json_mock['duration'] / 60)
print(f"Quality metrics from verbose response:")
print(f"  Average no-speech probability: {avg_no_speech:.3f} (< 0.1 = clear speech)")
print(f"  Speaking rate: {words_per_minute:.0f} words/minute")

In [ ]:
# Batch simulation — run the production classifier + router on all 5 mock transcripts
# It skips repeated transcription, but validates the same source classification function.

print("=== Batch Simulation: All 5 Intent Types ===")
print("(Uses mock transcripts — skips Whisper to avoid repeated API cost)\n")

from src.tools import classify_and_extract, route

batch_results = []

for intent_label, data in INTENT_EXAMPLES.items():
    transcript = data["transcript"]
    analysis = classify_and_extract(transcript, client, model="gpt-5.4-nano")

    actual_intent = analysis.get("intent", "general")
    actual_queue = route(actual_intent)
    expected_queue = data["expected_route"]
    match = "PASS" if actual_queue == expected_queue else "FAIL"

    batch_results.append({
        "label": intent_label,
        "actual_intent": actual_intent,
        "actual_queue": actual_queue,
        "expected_queue": expected_queue,
        "match": match,
    })

print(f"{'Label':<14} {'Predicted':<14} {'Queue':<22} {'Expected':<22} {'Match'}")
print("-" * 80)
for r in batch_results:
    print(f"{r['label']:<14} {r['actual_intent']:<14} {r['actual_queue']:<22} {r['expected_queue']:<22} {r['match']}")

passes = sum(1 for r in batch_results if r["match"] == "PASS")
print(f"\nRouting accuracy: {passes}/{len(batch_results)} ({100*passes//len(batch_results)}%)")

---
## What's Next

- **Whisper paper (OpenAI, 2022):** "Robust Speech Recognition via Large-Scale Weak Supervision" — [arxiv:2212.04356](https://arxiv.org/abs/2212.04356). Covers the architecture, training data, and multi-task formulation behind Whisper.
- **AssemblyAI LeMUR:** [assemblyai.com/docs/models/lemur](https://www.assemblyai.com/docs/models/lemur) — LLM-powered audio analysis combining transcription and custom prompts in one API call.
- **Real-time streaming transcription:** Whisper's streaming mode with `pyaudio` + websockets enables sub-second partial transcripts. Combine with the analyze/route nodes for live call monitoring.
- **Speaker diarization:** Combine Whisper with [pyannote.audio](https://github.com/pyannote/pyannote-audio) to label who said what in multi-speaker calls.
- **Example 138 (vision-qa-agent):** The same three-node `StateGraph` pattern applied to image understanding — same architecture, different modality.